# grad-accumulate-on-leaf — faded example 1: Faded: implement the first-touch branch of accumulate_grad

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `grad-accumulate-on-leaf`. Running the beacon reports progress on the `Backprop: Grad accumulate on leaf` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Grad accumulate on leaf` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grad-accumulate-on-leaf`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grad-accumulate-on-leaf"
DD_SUBTOPIC = "Backprop: Grad accumulate on leaf"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Gradient accumulation on a leaf uses a two-branch rule. On the FIRST backward visit, `leaf.grad` is `None` — we must set it directly to the incoming gradient. On all subsequent visits, we ADD to the existing gradient using rebinding (`leaf.grad = leaf.grad + g`), not in-place `+=`. Getting the first-touch branch right avoids adding to `None` (which would raise a TypeError).

## Faded exercise 1

Complete the `accumulate_grad` function below. The function must:
- Set `leaf.grad = g` when `leaf.grad is None` (first touch).
- Otherwise add: `leaf.grad = leaf.grad + g`.

The blank is specifically the expression you assign to `leaf.grad` when it IS already set (the else branch).

**Fill in:** The accumulation expression that rebinds leaf.grad to the sum of the current grad and the incoming gradient g.

In [ ]:
import torch as t

t.manual_seed(0)

class SimpleLeaf:
    def __init__(self):
        self.grad = None

def accumulate_grad(leaf, g):
    if leaf.grad is None:
        leaf.grad = g
    else:
        leaf.grad = None  # TODO: The accumulation expression that rebinds leaf.grad to the sum of the current grad and the incoming gradient g.

# Exercise it
leaf = SimpleLeaf()
g1 = t.tensor([1.0, 2.0])
g2 = t.tensor([3.0, 4.0])
accumulate_grad(leaf, g1)
accumulate_grad(leaf, g2)
print(leaf.grad)  # should be [4.0, 6.0]


def _test():
    import torch as t

    class SimpleLeaf:
        def __init__(self):
            self.grad = None

    leaf = SimpleLeaf()
    g1 = t.tensor([1.0, 2.0])
    g2 = t.tensor([3.0, 4.0])
    g3 = t.tensor([-1.0, 1.0])

    # Ground truth: simple manual sum
    accumulate_grad(leaf, g1)
    assert leaf.grad is not None
    assert t.allclose(leaf.grad, g1), f"After first call, expected {g1}, got {leaf.grad}"

    accumulate_grad(leaf, g2)
    assert t.allclose(leaf.grad, g1 + g2), f"After second call, expected {g1+g2}, got {leaf.grad}"

    # Rebind check: old reference must not have been mutated
    old_ref = leaf.grad
    accumulate_grad(leaf, g3)
    assert t.allclose(leaf.grad, g1 + g2 + g3)
    assert t.allclose(old_ref, g1 + g2), "in-place += detected: old reference was mutated"


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

class SimpleLeaf:
    def __init__(self):
        self.grad = None

def accumulate_grad(leaf, g):
    if leaf.grad is None:
        leaf.grad = g
    else:
        leaf.grad = leaf.grad + g

# Exercise it
leaf = SimpleLeaf()
g1 = t.tensor([1.0, 2.0])
g2 = t.tensor([3.0, 4.0])
accumulate_grad(leaf, g1)
accumulate_grad(leaf, g2)
print(leaf.grad)  # should be [4.0, 6.0]
```
</details>